# Tox21 x Mass Spec: Dataset Construction + Evaluation

Builds train/val/test splits, trains RF classifiers on compounds *without* spectra,
then evaluates both ground-truth MACCS and MS2MACCS-predicted MACCS on the held-out splits.

In [1]:
import numpy as np
import pandas as pd
import joblib

from rdkit import Chem
from rdkit.Chem import MACCSkeys, inchi as rdinchi

from matchms.importing import load_from_mgf

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score

from tqdm import tqdm

import sys
sys.path.append("../../")
sys.path.append("../../scripts")
from ms2maccs import MS2MACCS

## 1  Load raw data

In [2]:
tox21 = pd.read_csv("../raw_data/tox21.csv")
SPECTRAVERSE = "../raw_data/spectraverse_most_frags_processed.mgf"

VAL_MGF  = "../../ms2_data/val_specs_H_p_mode.mgf"
TEST_MGF = "../../ms2_data/test_specs_H_p_mode.mgf"

TOX_COLS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase",
    "NR-ER", "NR-ER-LBD", "NR-PPAR-gamma",
    "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53",
]

tox21.head()

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


## 2  Build InChIKey index for tox21

In [3]:
tox21["inchikey"] = [
    rdinchi.MolToInchiKey(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) is not None else np.nan
    for s in tqdm(tox21.smiles, desc="tox21 inchikeys")
]
tox21 = tox21.dropna(subset=["inchikey"])

tox21 inchikeys:  27%|███████████████▊                                           | 2105/7831 [00:01<00:02, 1944.43it/s][15:03:11] Explicit valence for atom # 3 Al, 6, is greater than permitted
[15:03:11] Explicit valence for atom # 4 Al, 6, is greater than permitted
tox21 inchikeys:  58%|██████████████████████████████████▏                        | 4542/7831 [00:02<00:01, 1920.56it/s][15:03:12] Explicit valence for atom # 9 Al, 6, is greater than permitted
[15:03:12] Explicit valence for atom # 5 Al, 6, is greater than permitted
tox21 inchikeys: 100%|███████████████████████████████████████████████████████████| 7831/7831 [00:04<00:00, 1920.54it/s]


In [4]:
tox21 = tox21.set_index("mol_id")
inchikey_to_molid = {row.inchikey: mol_id for mol_id, row in tox21.iterrows()}

In [5]:
tox21.shape

(7823, 14)

## 3  Match Spectraverse spectra to tox21 compounds

**Fixes vs original:**
- Filter to positive ionmode *before* the tox21 match
- Skip non-matching spectra entirely (original appended all of them)
- Tag matched spectra with `mol_id` for later lookup
- `continue` in the original never broke out of the inner loop; now we use a dict lookup (O(1))
- Deduplication now guards against `mol_id=None` (original collapsed all non-matching specs into one slot)

In [6]:
unique_tox_spectra = []

for spec in tqdm(load_from_mgf(SPECTRAVERSE), desc="Spectraverse"):
    if spec.get("ionmode") != "positive":
        continue
    ik = spec.get("inchikey")
    mol_id = inchikey_to_molid.get(ik)   # None if not a tox21 compound
    if mol_id is None:
        continue                          # skip non-tox21 spectra entirely
    spec.set("mol_id", str(mol_id))
    unique_tox_spectra.append(spec)

print(f"Unique tox21 compounds:  {len(unique_tox_spectra)}")

Spectraverse: 44297it [00:32, 1371.62it/s]

Unique tox21 compounds:  2237


## 4  Identify train / val / test compounds

`tox21_withSpec` = compounds that appear in the pre-built val or test MGF files (held-out).  
`tox21_noSpec`   = everything else - used to train the RF classifiers.

In [7]:
val_specs  = list(load_from_mgf(VAL_MGF))
test_specs = list(load_from_mgf(TEST_MGF))

val_mol_ids  = {inchikey_to_molid[s.get("inchikey")]
                for s in val_specs  if s.get("inchikey") in inchikey_to_molid}
test_mol_ids = {inchikey_to_molid[s.get("inchikey")]
                for s in test_specs if s.get("inchikey") in inchikey_to_molid}
held_out_ids = val_mol_ids | test_mol_ids

tox21_withSpec = tox21[tox21.index.isin(held_out_ids)]
tox21_noSpec   = tox21[~tox21.index.isin(held_out_ids)]

print(f"With spectrum (held-out): {len(tox21_withSpec)}")
print(f"Without spectrum (train): {len(tox21_noSpec)}")

# Save for reference (fix: add .csv extension)
tox21_withSpec.to_csv("tox21_withSpec.csv")

With spectrum (held-out): 1738
Without spectrum (train): 6085


## 5  Compute MACCS keys for the training set

In [8]:
tox21_noSpec = tox21_noSpec.copy()
tox21_noSpec["maccs"] = [
    np.array(MACCSkeys.GenMACCSKeys(Chem.MolFromSmiles(s)))
    for s in tqdm(tox21_noSpec.smiles, desc="MACCS (train)")
]

MACCS (train): 100%|██████████████████████████████████████████████████████████████| 6085/6085 [00:07<00:00, 788.52it/s]


## 6  Train one Random Forest classifier per tox endpoint

In [9]:
models = {}
for col in TOX_COLS:
    subset = tox21_noSpec.dropna(subset=[col])
    X = np.vstack(subset["maccs"].tolist())
    y = subset[col].tolist()
    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    rf.fit(X, y)
    models[col] = rf
    print(f"Trained {col:20s}  n={len(y)}")

# Persist to disk
joblib_names = {
    "NR-AR": "rf_nr_ar", "NR-AR-LBD": "rf_nr_ar_lbd", "NR-AhR": "rf_nr_ahr",
    "NR-Aromatase": "rf_nr_aromatase", "NR-ER": "rf_nr_er", "NR-ER-LBD": "rf_nr_er_lbd",
    "NR-PPAR-gamma": "rf_nr_ppar_gamma", "SR-ARE": "rf_sr_are", "SR-ATAD5": "rf_sr_atad5",
    "SR-HSE": "rf_nr_hse", "SR-MMP": "rf_nr_mmp", "SR-p53": "rf_nr_p53",
}
for col, name in joblib_names.items():
    joblib.dump(models[col], f"{name}.joblib")

Trained NR-AR                 n=5596
Trained NR-AR-LBD             n=5204
Trained NR-AhR                n=5063
Trained NR-Aromatase          n=4500
Trained NR-ER                 n=4796
Trained NR-ER-LBD             n=5369
Trained NR-PPAR-gamma         n=4972
Trained SR-ARE                n=4421
Trained SR-ATAD5              n=5443
Trained SR-HSE                n=4912
Trained SR-MMP                n=4512
Trained SR-p53                n=5225


## 7  Load MS2MACCS model

In [10]:
m = MS2MACCS(
    "../../models/standard_model.pt",
    "../../fp_bit_maps/fp_bit_map_H_p_mode.pkl",
    "../../fp_bit_maps/fp_bit_map_H_n_mode.pkl",
    "cpu",  # replace with "cpu" if no GPU
)

## 8  Evaluation helper

For each split:
1. Extract ground-truth MACCS from the `smiles` field in each spectrum's metadata
2. Call `MS2MACCS.predict()` on the MGF to get predicted MACCS (binarised at 0.5)
3. Align tox21 labels to spectrum order via inchikey
4. Evaluate balanced accuracy for both MACCS sources per endpoint

In [24]:
def evaluate_split(mgf_path: str, specs: list, split_name: str):
    """Evaluate RF tox predictions on a pre-loaded spectrum list.

    Args:
        mgf_path: path to the MGF file (passed to MS2MACCS.predict)
        specs:    already-loaded list of matchms Spectrum objects
        split_name: label for printing
    """
    print(f"\n{split_name}: {len(specs)} spectra")

    # 1. Ground-truth MACCS from the SMILES stored inside each spectrum
    gt_maccs = np.vstack([
        np.array(MACCSkeys.GenMACCSKeys(Chem.MolFromSmiles(s.get("smiles"))))
        for s in specs
    ])

    # 2. MS2MACCS-predicted MACCS
    pred_maccs = (m.predict(mgf_path).to("cpu").numpy() > 0.5).astype(int)

    # 3. Tox labels aligned to spectrum order
    mol_ids    = [inchikey_to_molid.get(s.get("inchikey")) for s in specs]
    tox_labels = tox21.reindex(mol_ids)[TOX_COLS].values  # (n, 12), NaN where absent

    # 4. Binary tox vectors (replace NaN with 0, then convert to binary)
    binary_tox_labels = np.where(np.isnan(tox_labels), 0, tox_labels).astype(int)

    # 5. Balanced accuracy per endpoint
    results_gt, results_pred = {}, {}
    for i, col in enumerate(TOX_COLS):
        mask = ~np.isnan(tox_labels[:, i])
        if mask.sum() == 0:
            continue
        y = tox_labels[mask, i]
        results_gt[col]   = balanced_accuracy_score(y, models[col].predict(gt_maccs[mask]))
        results_pred[col] = balanced_accuracy_score(y, models[col].predict(pred_maccs[mask]))
        print(f"  {col:20s}  gt={results_gt[col]:.4f}  ms2maccs={results_pred[col]:.4f}")

    return results_gt, results_pred, gt_maccs, pred_maccs, binary_tox_labels

## 9  Run evaluation

In [25]:
val_gt,  val_pred, val_maccs_gt, val_maccs_pred, val_tox = evaluate_split(VAL_MGF,  val_specs,  "Validation")
test_gt, test_pred, test_maccs_gt, test_maccs_pred, test_tox = evaluate_split(TEST_MGF, test_specs, "Test")


Validation: 4409 spectra


Processing val_specs_H_p_mode.mgf: 4409it [00:09, 451.38it/s]
Prediction: 100%|██████████████████████████████████████████████████████████████████| 4409/4409 [01:53<00:00, 38.99it/s]


  NR-AR                 gt=0.5131  ms2maccs=0.4984
  NR-AR-LBD             gt=0.4992  ms2maccs=0.4996
  NR-AhR                gt=0.6602  ms2maccs=0.5563
  NR-Aromatase          gt=0.5422  ms2maccs=0.5000
  NR-ER                 gt=0.5874  ms2maccs=0.5115
  NR-ER-LBD             gt=0.5583  ms2maccs=0.4996
  NR-PPAR-gamma         gt=0.4995  ms2maccs=0.5000
  SR-ARE                gt=0.5544  ms2maccs=0.5084
  SR-ATAD5              gt=0.5362  ms2maccs=0.5093
  SR-HSE                gt=0.5358  ms2maccs=0.5000
  SR-MMP                gt=0.6535  ms2maccs=0.5181
  SR-p53                gt=0.5413  ms2maccs=0.5000

Test: 1484 spectra


Processing test_specs_H_p_mode.mgf: 1484it [00:03, 427.62it/s]
Prediction: 100%|██████████████████████████████████████████████████████████████████| 1484/1484 [00:37<00:00, 39.21it/s]


  NR-AR                 gt=0.4987  ms2maccs=0.4974
  NR-AR-LBD             gt=0.5819  ms2maccs=0.4986
  NR-AhR                gt=0.6577  ms2maccs=0.5624
  NR-Aromatase          gt=0.5203  ms2maccs=0.5000
  NR-ER                 gt=0.5864  ms2maccs=0.5239
  NR-ER-LBD             gt=0.6072  ms2maccs=0.5000
  NR-PPAR-gamma         gt=0.5357  ms2maccs=0.5000
  SR-ARE                gt=0.6081  ms2maccs=0.5133
  SR-ATAD5              gt=0.5371  ms2maccs=0.5200
  SR-HSE                gt=0.5556  ms2maccs=0.5000
  SR-MMP                gt=0.7195  ms2maccs=0.5393
  SR-p53                gt=0.5117  ms2maccs=0.5000


## 10  Summary table

In [26]:
summary = pd.DataFrame({
    "Val  - gt MACCS":   val_gt,
    "Val  - MS2MACCS":   val_pred,
    "Test - gt MACCS":   test_gt,
    "Test - MS2MACCS":   test_pred,
})
summary.index.name = "Endpoint"
summary.round(4)

,Val - gt MACCS,Val - MS2MACCS,Test - gt MACCS,Test - MS2MACCS
Endpoint,,,,
NR-AR,0.5131,0.4984,0.4987,0.4974
NR-AR-LBD,0.4992,0.4996,0.5819,0.4986
NR-AhR,0.6602,0.5563,0.6577,0.5624
NR-Aromatase,0.5422,0.5000,0.5203,0.5000
NR-ER,0.5874,0.5115,0.5864,0.5239
NR-ER-LBD,0.5583,0.4996,0.6072,0.5000
NR-PPAR-gamma,0.4995,0.5000,0.5357,0.5000
SR-ARE,0.5544,0.5084,0.6081,0.5133
SR-ATAD5,0.5362,0.5093,0.5371,0.5200


## 11 FP comparison

In [22]:
def calc_tanimoto(fp1, fp2):
    set1 = set(fp1) if isinstance(fp1, (list, set)) else set(np.where(fp1)[0])
    set2 = set(fp2) if isinstance(fp2, (list, set)) else set(np.where(fp2)[0])

    intersection = len(set1 & set2)
    union = len(set1 | set2)

    return intersection / union if union != 0 else 0.0

In [ ]:
def find_different_bits(fp1, fp2):
    """Return the indices of bits that differ between two binary fingerprints."""
    return np.where(fp1 != fp2)[0]


for i in range(12):
    count_wrong = 0
    count_correct = 0
    for gt, pred, tox in zip(val_maccs_gt, val_maccs_pred, val_tox):
        tox_pred_gt = models[TOX_COLS[i]].predict(gt.reshape(1, -1)) 
        tox_pred_ms2maccs = models[TOX_COLS[i]].predict(pred.reshape(1, -1))
    
        if tox_pred_gt != tox_pred_ms2maccs:
            # If predictions differ, find differing bits
            different_bits = find_different_bits(gt, pred)
            print(f"Predictions differ. Different bits: {different_bits} Tanimoto: {calc_tanimoto(gt, pred)}")
            count_wrong += 1
        else:
            # If predictions agree, just print Tanimoto
            #print(calc_tanimoto(gt, pred))
            count_correct += 1

    print("***************************")
    print(count_wrong, count_correct)
    print("***************************")

Predictions differ. Different bits: [ 26  72  82 126 136 150 153 155] Tanimoto: 0.84
Predictions differ. Different bits: [ 19  26  89 113 129 131 145 162] Tanimoto: 0.8260869565217391
Predictions differ. Different bits: [ 89  96 104 123 131 153] Tanimoto: 0.8604651162790697
Predictions differ. Different bits: [ 26  74  82 104 136 138] Tanimoto: 0.8888888888888888
Predictions differ. Different bits: [ 26  72  82 104 126 136] Tanimoto: 0.8723404255319149
Predictions differ. Different bits: [136 162] Tanimoto: 0.9636363636363636
Predictions differ. Different bits: [ 26  34  66  74  83  89 104 112 132 141 145 155 163] Tanimoto: 0.717391304347826
Predictions differ. Different bits: [ 26  34  83  91 104 118 128 136 147] Tanimoto: 0.7954545454545454
Predictions differ. Different bits: [ 26  82  83  90  91  98 104 109 120 123 131 136 155] Tanimoto: 0.7450980392156863
Predictions differ. Different bits: [ 26  50  57  75  76  80  90  91  95 108 109 116 125 141 151] Tanimoto: 0.7887323943661971
P